<h1 style='font-weight:bold; color:#003366; text-align:center;'>Práctico 1 : Clean Code, linting y funciones</h1>

<h3 style='font-weight:bold; color:#003366; text-align:center;'>Programación Analítica II</h3>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>Docente Camila Palomeque</h4>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>S2 . 2026</h4>

| | |
|---|---|
| **Temas evaluados** | Temas 1, 2 y 3 |
| **Fecha de entrega** | Martes 1 de setiembre, 23:59 |
| **Dataset** | `transacciones_personales.csv` |


Los ejercicios están ordenados de menor a mayor dificultad y **cada uno usa lo que
construiste en el anterior** por lo que es necesario ir en orden.

<h2 style='font-weight:bold; color:#003366;'>Instrucciones de entrega</h2>

1. Nombrá el archivo `Practico1_ApellidoNombre.ipynb`, con tu apellido y tu nombre.
2. Subilo a la tarea correspondiente del Moodle antes de la fecha límite.
3. **Ejecutá todas las celdas de arriba abajo antes de entregar** y verificá que no haya errores.
   Un notebook que no corre pierde puntos, sin importar qué diga el código.
   En Jupyter: *Kernel → Restart Kernel and Run All Cells*.
4. Las respuestas van en las celdas de **texto** (markdown) que dicen `_(tu respuesta acá)_`.
   No en comentarios del código.
5. El código tiene que cumplir con lo visto en los temas 1 y 2: nombres descriptivos, funciones
   cortas, constantes con nombre en lugar de números sueltos, y estilo PEP 8.

<div style="border-left: 5px solid #BB3300; padding: 12px 16px; background-color: #FFF3F0;">
<b style="color:#BB3300;">Sobre el uso de asistentes de inteligencia artificial</b><br><br>
Podés usarlos como apoyo para entender conceptos o depurar errores. Lo que <b>no</b> es aceptable es entregar código que no podés explicar.<br><br>
En la corrección en clase se pueden hacer preguntas sobre cualquier línea de tu entrega. Si no podés justificar una decisión,
esa parte se considera no cumplida. <br><br>
El objetivo del práctico es que aprendas las estructuras básicas de programación analítica en python, 
lo cual te va a servir para el parcial, no que se produzca un archivo.<br><br>
</div>

<h2 style='font-weight:bold; color:#003366;'>Antes de arrancar</h2>

Corré estas dos celdas. La primera instala las herramientas de la parte 2 si te faltan, la segunda
encuentra el archivo de datos sin importar si estás en Colab o en tu máquina.

In [1]:
# Celda de verificación inicial: corré esta celda primero.
# Si estás en Colab instala las herramientas, si estás en local y ejecutaste el notebook del tema 2 ya las tenés.
# De lo contrario con esta celda se instalan automáticamente.
from importlib.util import find_spec

HERRAMIENTAS = ["flake8", "black", "pylint"]
faltantes = [nombre for nombre in HERRAMIENTAS if find_spec(nombre) is None]

if faltantes:
    paquetes = " ".join(faltantes)
    print("Instalando:", paquetes)
    !pip install -q {paquetes}
else:
    print("flake8, black y pylint ya están instaladas.")

flake8, black y pylint ya están instaladas.


In [2]:
import sys
from pathlib import Path


def ruta_datos(nombre_archivo):
    """Busca el archivo en las ubicaciones habituales del curso"""
    candidatas = [
        Path("../Datos") / nombre_archivo,
        Path("Datos") / nombre_archivo,
        Path("../datos") / nombre_archivo,
        Path("datos") / nombre_archivo,
        Path("/content/Datos") / nombre_archivo,
        Path("/content") / nombre_archivo,
        Path(nombre_archivo),
    ]
    for candidata in candidatas:
        if candidata.exists():
            return str(candidata)
    raise FileNotFoundError(f"No se encontró {nombre_archivo}.")


print("Python", sys.version.split()[0])
print("Archivo de datos:", ruta_datos("transacciones_personales.csv"))

Python 3.13.9
Archivo de datos: transacciones_personales.csv


<h2 style='font-weight:bold; color:#003366;'>Parte 1 : refactorizar código ajeno <span style='color:#BB3300; font-size:0.8em;'>(20 puntos)</span></h2>

Un compañero del área de Finanzas te pasa esta función. **Funciona**: corré la celda y vas a
ver que imprime números.

Tu trabajo es dejarla entendible sin cambiar los números que produce.

In [3]:
# Código a refactorizar. Corré la celda para ver qué imprime.
# NO lo edites acá: escribí tu versión en la celda del ejercicio 1.1.

MOVS = [
    ("2026-03-01", 52000.0, "Ingresos"),
    ("2026-03-03", -14500.0, "Vivienda"),
    ("2026-03-05", -6240.0, "Supermercado"),
    ("2026-03-09", -4200.0, "Salud"),
    ("2026-03-12", -9800.0, "Educacion"),
    ("2026-03-15", -390.0, "Ocio"),
    ("2026-03-18", -3100.0, "Transporte"),
    ("2026-03-22", 8000.0, "Ingresos"),
    ("2026-03-26", -1850.0, "Ocio"),
]


def proc(l):
    # Devuelve el promedio de los ingresos del período
    t = 0
    n = 0
    for i in range(len(l)):
        if l[i][1] < 0:
            t = t + l[i][1]
            n = n + 1
    e = 0
    for i in range(len(l)):
        if l[i][1] < 0 and l[i][2] in ["Vivienda", "Salud", "Supermercado"]:
            e = e + l[i][1]
    r = 0
    for i in range(len(l)):
        if l[i][1] > 0:
            r = r + l[i][1]
    print("Gasto promedio:", t / n)
    print("Gasto esencial:", e)
    if (r + t) / r < 0.2:
        print("Alerta: la tasa de ahorro quedo abajo del minimo")
    return t / n, e, r, (r + t) / r


proc(MOVS)

Gasto promedio: -5725.714285714285
Gasto esencial: -24940.0


(-5725.714285714285, -24940.0, 60000.0, 0.332)

<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 1.1 <span style='color:#BB3300; font-size:0.8em;'>* (15 pts)</span></h3>

Reescribí ese código. Requisitos:

- **Los cuatro números que devuelve tienen que ser los mismos.** Dejá una celda que compare tu
  resultado contra el original y lo confirme.
- Partilo en funciones cortas, con una responsabilidad cada una y un nombre que empiece con verbo.
- Los valores fijos (el umbral de la alerta, la lista de categorías esenciales) van como constantes
  con nombre, arriba.

In [4]:
CATEGORIAS_ESENCIALES = {"Vivienda", "Salud", "Supermercado"}
UMBRAL_ALERTA = 0.20


def calcular_gasto_promedio(movimientos):
    """Calcula el monto promedio de las transacciones clasificadas como gastos (monto negativo)."""
    gastos = [monto for _, monto, _ in movimientos if monto < 0]
    return sum(gastos) / len(gastos) if gastos else 0.0


def calcular_gasto_esencial(movimientos):
    """Suma los montos de gastos correspondientes a categorías esenciales."""
    return sum(
        monto
        for _, monto, categoria in movimientos
        if monto < 0 and categoria in CATEGORIAS_ESENCIALES
    )


def calcular_total_ingresos(movimientos):
    """Suma todos los montos positivos correspondientes a ingresos."""
    return sum(monto for _, monto, _ in movimientos if monto > 0)


def calcular_tasa_ahorro(total_ingresos, total_gastos):
    """Calcula la relación entre el saldo neto (ingresos + gastos [son negativos]) y los ingresos totales."""
    if total_ingresos == 0:
        return 0.0
    return (total_ingresos + total_gastos) / total_ingresos


def analizar_finanzas_personales(movimientos):
    """Analiza los movimientos financieros del período y genera las métricas clave. Retorna una tupla"""
    gasto_promedio = calcular_gasto_promedio(movimientos)
    gasto_esencial = calcular_gasto_esencial(movimientos)
    total_ingresos = calcular_total_ingresos(movimientos)

    # La suma de todos los montos negativos es la cantidad total gastada
    total_gastos = sum(monto for _, monto, _ in movimientos if monto < 0)
    tasa_ahorro = calcular_tasa_ahorro(total_ingresos, total_gastos)

    # Salida por pantalla separada de los cálculos puros
    print(f"Gasto promedio: {gasto_promedio}")
    print(f"Gasto esencial: {gasto_esencial}")

    if tasa_ahorro < UMBRAL_ALERTA:
        print("Alerta: la tasa de ahorro quedo abajo del minimo")

    return gasto_promedio, gasto_esencial, total_ingresos, tasa_ahorro


# Verificación para ver si los resultados son iguales
resultado_original = proc(MOVS)
resultado_refactorizado = analizar_finanzas_personales(MOVS)

print("\nVerificación:")
print("Resultado original:   ", resultado_original)
print("Resultado nuevo:      ", resultado_refactorizado)
print("¿Coinciden exactamente?:", resultado_original == resultado_refactorizado)

Gasto promedio: -5725.714285714285
Gasto esencial: -24940.0
Gasto promedio: -5725.714285714285
Gasto esencial: -24940.0

Verificación:
Resultado original:    (-5725.714285714285, -24940.0, 60000.0, 0.332)
Resultado nuevo:       (-5725.714285714285, -24940.0, 60000.0, 0.332)
¿Coinciden exactamente?: True


<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 1.2 <span style='color:#BB3300; font-size:0.8em;'>* (5 pts)</span></h3>

Contestá abajo explicando qué cambios hiciste. 
En una fila por cambio respondé: Descripción del cambio, qué principio de Clean Code aplica, qué mejora para el que lo lee.


### Ejercicio 1.2 - Justificación de Cambios y Principios de Clean Code

| Descripción del cambio | Principio de Clean Code aplicado | Qué mejora para el que lo lee |
|---|---|---|
| **Nombres descriptivos y semánticos** (ej., `movimientos`, `gasto_promedio` en lugar de `l`, `t`, `n`). | 2. Nombres descriptivos | Evita ejecutar mentalmente el código y descifrar el significado de variables opacas de una sola letra. |
| **Definición de constantes** (`CATEGORIAS_ESENCIALES`, `UMBRAL_TASA_AHORRO_MINIMA`). | 8. Números mágicos y constantes con nombre | Otorga contexto de negocio a valores literales sueltos y centraliza cualquier cambio futuro de parámetros. |
| **Modularización en funciones cortas** (`calcular_gasto_promedio`, `calcular_gasto_esencial`, etc.). | 3. Una función, una sola responsabilidad | Permite probar y reutilizar cada cálculo financiero por separado sin arrastrar impresiones ni bucles acoplados. |
| **Uso de comprensiones de listas, desempaquetado de tuplas y `set`** en vez de iterar por índice (`range(len(l))`). | 10. Comprensiones de listas | Reemplaza bucles `for` repetitivos por iteraciones directas, haciendo el código más rápido y fácil de leer. |
| **Inclusión de Docstrings** en cada función. | 4. Comentarios y docstrings | Dice el propósito de la función sin saturar con comentarios redundantes. |
| **Formateo y espaciado de acuerdo a PEP 8**. | 11. PEP 8 : la guía de estilo de Python | Homogeniza la apariencia visual del código, facilitando una lectura fluida y estandarizada. |

<h2 style='font-weight:bold; color:#003366;'>Parte 2 : linting <span style='color:#BB3300; font-size:0.8em;'>(10 puntos)</span></h2>

Orden de las herramientas, siempre el mismo:

1. `black` formatea
2. `flake8` encuentra errores
3. `pylint` opina sobre el diseño

En esta parte no escribís código nuevo: agarrás un archivo sucio que te damos y lo dejás limpio.

<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 2.1 <span style='color:#BB3300; font-size:0.8em;'>* (6 pts)</span></h3>

Corré las dos celdas que siguen. La primera crea el archivo, la segunda lo pasa por `flake8`.

Después contestá, en la celda de texto: **¿cuántos mensajes distintos encontró `flake8`?**
Y comenta, al pasar el código por `black`, cuáles **no** va a poder arreglar y por qué.

In [5]:
%%writefile presupuesto_sucio.py
import os
import math

def CalcularPresupuesto(sueldo,gastos_fijos ):
    total = 0
    for g in gastos_fijos :
        total=total+g
    disponible = sueldo-total
    aux = sueldo * 0.15
    if disponible < sueldo*0.15 : print("Alerta: queda menos del 15 por ciento del sueldo disponible para el resto del mes")
    return disponible, disponible/sueldo

SUELDO_NOMINAL = 52000
GASTOS_DEL_MES = [14500, 6240, 4200, 9800, 3100]
print(CalcularPresupuesto(SUELDO_NOMINAL, GASTOS_DEL_MES))

Overwriting presupuesto_sucio.py


In [6]:
!python -m flake8 --max-line-length=88 presupuesto_sucio.py

presupuesto_sucio.py:1:1: F401 'os' imported but unused
presupuesto_sucio.py:2:1: F401 'math' imported but unused
presupuesto_sucio.py:4:1: E302 expected 2 blank lines, found 1
presupuesto_sucio.py:4:31: E231 missing whitespace after ','
presupuesto_sucio.py:4:44: E202 whitespace before ')'
presupuesto_sucio.py:6:26: E203 whitespace before ':'
presupuesto_sucio.py:7:14: E225 missing whitespace around operator
presupuesto_sucio.py:9:5: F841 local variable 'aux' is assigned to but never used
presupuesto_sucio.py:10:32: E203 whitespace before ':'
presupuesto_sucio.py:10:33: E701 multiple statements on one line (colon)
presupuesto_sucio.py:10:89: E501 line too long (124 > 88 characters)
presupuesto_sucio.py:13:1: E305 expected 2 blank lines after class or function definition, found 1


_(tu respuesta acá)_

Flake8 reportó 10 códigos de error distintos (con algunos repetidos en varias líneas): 
- F401, E302, E231, E202, E203, E225, F841, E701, E501 y E305.  

Los mensajes que black no va a poder arreglar son:
- F401 (os y math importados sin usar): black nunca elimina importaciones ni código.  
- F841 (aux asignada y no usada): black no elimina variables inactivas o código muerto.  
- E501 (línea demasiado larga): si la longitud de la cadena de texto por sí sola supera los 88 caracteres, black no divide el el texto del string para evitar alterar los datos.

Corré ahora sobre el código `black --diff` para ver qué cambiaría, después aplicalo y volvé a correr `flake8`.

Confirma **¿qué códigos desaparecieron después de `black` y cuáles quedaron?**

In [7]:
# Aplica black (con --diff y --line-length 88) sobre presupuesto_sucio.py
!python -m black --diff --line-length 88 presupuesto_sucio.py

--- presupuesto_sucio.py	2026-09-01 22:32:15.082404+00:00
+++ presupuesto_sucio.py	2026-09-01 22:32:16.379197+00:00
@@ -1,15 +1,20 @@
 import os
 import math
 
-def CalcularPresupuesto(sueldo,gastos_fijos ):
+
+def CalcularPresupuesto(sueldo, gastos_fijos):
     total = 0
-    for g in gastos_fijos :
-        total=total+g
-    disponible = sueldo-total
+    for g in gastos_fijos:
+        total = total + g
+    disponible = sueldo - total
     aux = sueldo * 0.15
-    if disponible < sueldo*0.15 : print("Alerta: queda menos del 15 por ciento del sueldo disponible para el resto del mes")
-    return disponible, disponible/sueldo
+    if disponible < sueldo * 0.15:
+        print(
+            "Alerta: queda menos del 15 por ciento del sueldo disponible para el resto del mes"
+        )
+    return disponible, disponible / sueldo
+
 
 SUELDO_NOMINAL = 52000
 GASTOS_DEL_MES = [14500, 6240, 4200, 9800, 3100]
 print(CalcularPresupuesto(SUELDO_NOMINAL, GASTOS_DEL_MES))


would reformat presupuesto_sucio.py

All done! ✨ 🍰 ✨
1 file would be reformatted.


In [8]:
# Aplica black (sin --diff) y volve a correr flake8
!python -m black --line-length 88 presupuesto_sucio.py
!python -m flake8 --max-line-length=88 presupuesto_sucio.py

reformatted presupuesto_sucio.py

All done! ✨ 🍰 ✨
1 file reformatted.


presupuesto_sucio.py:1:1: F401 'os' imported but unused
presupuesto_sucio.py:2:1: F401 'math' imported but unused
presupuesto_sucio.py:10:5: F841 local variable 'aux' is assigned to but never used
presupuesto_sucio.py:13:89: E501 line too long (95 > 88 characters)


_(tu respuesta acá)_

Desaparecieron todas las alertas de formato e espaciado de flake8 de la corrida del Ejercicio 2.1:
- E302 y E305 (líneas en blanco faltantes antes/después de la función).
- E231, E202, E203 y E225 (espacios incorrectos alrededor de comas, paréntesis y operadores).
- E701 (múltiples sentencias en una sola línea en el if).

Códigos que quedaron:
- F401 (en 2 líneas): os y math importados sin usar.
- F841: variable local aux asignada pero no usada.
- E501: línea 13 demasiado larga (95 caracteres vs el límite de 88).

<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 2.3 <span style='color:#BB3300; font-size:0.8em;'>** (4 pts)</span></h3>

Ahora arreglá a mano lo que quedó, en la celda de abajo. Reescribí el archivo completo con
`%%writefile presupuesto_limpio.py` y dejalo:

- **sin ningún mensaje de `flake8`**, con `--max-line-length=88`
- **con puntaje de `pylint` de 8.00 o más**

Corré las dos herramientas sobre el archivo nuevo y dejá las salidas visibles.

Después contestá: de los mensajes de `pylint`, elegí **uno que decidiste no arreglar** y justificá
en dos oraciones por qué en este archivo no corresponde. 

Si `pylint` te dio 10.00 y no quedó ningún
mensaje, decilo y explicá qué tendrías que haber dejado en el archivo para que apareciera un
mensaje de `pylint` que `flake8` no detecta.

In [9]:
%%writefile presupuesto_limpio.py
"""Módulo para calcular el presupuesto disponible y el porcentaje sobre el sueldo nominal."""

PORCENTAJE_ALERTA = 0.15


def calcular_presupuesto(sueldo, gastos_fijos):
    """Calcula el monto disponible y la proporción sobre el sueldo nominal."""
    total_gastos = sum(gastos_fijos)
    disponible = sueldo - total_gastos
    if disponible < sueldo * PORCENTAJE_ALERTA:
        print(
            "Alerta: queda menos del 15 por ciento "
            "del sueldo disponible para el resto del mes"
        )
    return disponible, disponible / sueldo


SUELDO_NOMINAL = 52000
GASTOS_DEL_MES = [14500, 6240, 4200, 9800, 3100]
print(calcular_presupuesto(SUELDO_NOMINAL, GASTOS_DEL_MES))

Overwriting presupuesto_limpio.py


In [10]:
!python -m flake8 --max-line-length=88 presupuesto_limpio.py
!python -m pylint presupuesto_limpio.py

presupuesto_limpio.py:1:89: E501 line too long (93 > 88 characters)

--------------------------------------------------------------------
Your code has been rated at 10.00/10 (previous run: 10.00/10, +0.00)



_(tu respuesta acá)_

Flake8 me dejo un mensaje porque el docstring me quedó más largo de 88 caracteres, mientras que el límite por defecto de pylint es más grande.

Para que hubiese aparecido un mensaje de pylint que flake8 no detecta, tendría que haber usado un nombre de función en formato camelCase (como CalcularPresupuesto, o sea ponerle mayúsculas). flake8 no analiza reglas de nombres contra snake_case, mientras que pylint sí.

<h2 style='font-weight:bold; color:#003366;'>Parte 3 : funciones sobre datos reales <span style='color:#BB3300; font-size:0.8em;'>(55 puntos)</span></h2>

A partir de acá trabajás con `transacciones_personales.csv`: movimientos de una cuenta bancaria
personal, de enero de 2025 a junio de 2026. Las columnas son
`id`, `fecha`, `descripcion`, `monto`, `categoria` y `medio_pago`.

Dos cosas del archivo que ya sabés y que hay que tratar:

- Hay filas donde **el monto no vino**. El movimiento existe, el importe no. No es un movimiento de
  cero pesos.
- Algunas filas **no tienen categoría**, y la descripción del comercio sí está.

Todavía no usamos pandas. Todo se hace con listas, diccionarios y bucles.

<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 3.1 <span style='color:#BB3300; font-size:0.8em;'>* (5 pts)</span></h3>

La función `cargar_transacciones(ruta_archivo)` ya está escrita. Hace esto:

- Lee el CSV con `csv.DictReader`.
- Convierte `monto` a número.
- Descarta la fila si el monto no se puede convertir, usando `try` / `except`.
- Devuelve dos cosas: la lista de transacciones válidas y cuántas filas se descartaron.

En este ejercicio se pide:

1. Completa la llamada a la función cargar_transacciones para que lea las transacciones.
2. Si en lugar de descartarla las transacciones de las filas que no tenían monto las hubiera mantenido rellenando el monto con cero, **¿qué habría pasado con el
   promedio de gasto por movimiento?** (hubiera subido o bajado y por qué).

In [11]:
import csv

def cargar_transacciones(ruta_archivo):
    """Lee el CSV y devuelve las transacciones válidas y cuántas filas se descartaron."""
    transacciones = []
    descartadas = 0
    with open(ruta_archivo, encoding="utf-8") as archivo:
        for fila in csv.DictReader(archivo):
            try:
                fila["monto"] = float(fila["monto"])
                transacciones.append(fila)
            except (ValueError, TypeError):
                descartadas += 1
    return transacciones, descartadas


def sumar_montos(transacciones):
    """Suma la clave 'monto' de una lista de transacciones."""
    monto_total = 0.0
    for transaccion in transacciones:
        monto_total += transaccion["monto"]
    return monto_total

In [12]:
archivo = ruta_datos("transacciones_personales.csv")

# transacciones, descartadas = cargar_transacciones() # Arreglar llamada a la función para ejecutarla y cargar los datos
transacciones, descartadas = cargar_transacciones(archivo)

print("Transacciones cargadas:", len(transacciones))
print("Filas descartadas:", descartadas)
print("Primera transacción:", transacciones[0])

Transacciones cargadas: 348
Filas descartadas: 6
Primera transacción: {'id': '1', 'fecha': '2025-01-01', 'descripcion': 'Cuota universidad', 'monto': -1197.39, 'categoria': 'Educacion', 'medio_pago': 'Debito'}


_(tus dos respuestas acá)_

1. Llamada a la función: Se completa pasándole la variable archivo (que almacena la ruta obtenida con ruta_datos) como argumento a cargar_transacciones(archivo).

2. Efecto de meterle ceros en el promedio de gasto: El promedio habría bajado ya que al incluir observaciones con valor $0$ en el cálculo, el divisor del promedio ($N$, o total de elementos) aumenta mientras que la suma total acumulada de los montos ($S$) se mantiene igual. Ya que la media se calcula como $\frac{S}{N}$, aumentar la cantidad de transacciones sin sumar valor numérico achica el promedio general (a mayor divisor e igual numerador, menor el resultado). 

<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 3.2 <span style='color:#BB3300; font-size:0.8em;'>* (6 pts)</span></h3>

Escribí `filtrar_gastos(transacciones)`: que devuelve **solo** las transacciones cuyo monto es
negativo. 

La celda de abajo está empezada. Completá las líneas que dicen `...`.

Después usá `sumar_montos` (ya está escrita) para contestar: **¿cuántos egresos hay y cuánto suman
en total?** Muestra el total sin decimales, con signo de $ y en positivo.

In [13]:
def filtrar_gastos(transacciones):
    gastos = []
    for transaccion in transacciones:
        if transaccion["monto"] < 0:
            gastos.append(transaccion)
    return gastos


gastos = filtrar_gastos(transacciones)

# Imprimir cuántos son y cuánto suman
total_egresos = sumar_montos(gastos)
print(f"Cantidad de egresos: {len(gastos)}")
print(f"Total de gastos: ${abs(total_egresos):.0f}")

Cantidad de egresos: 324
Total de gastos: $695647


<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 3.3 <span style='color:#BB3300; font-size:0.8em;'>* (6 pts)</span></h3>

Escribí `contar_por_categoria(transacciones)`, esta funcion hace lo mismo que la vista en los ejercicios de clase pero esta resuelta sin usar get
completa los espacios sin cambiar la estructura de la funcion: devuelve un diccionario con **cuántas
transacciones** hay en cada categoría.

La celda está empezada. Completá las líneas que dicen `...`.

Mirá la salida de la función y contesta: **¿aparece alguna categoría rara?** .

In [14]:
def contar_por_categoria(transacciones):
    conteo = {}

    for transaccion in transacciones:
        categoria = transaccion["categoria"]

        if categoria in conteo:
            conteo[categoria] += 1
        else:
            conteo[categoria] = 1

    return conteo


conteo = contar_por_categoria(transacciones)
for categoria, cantidad in conteo.items():
    print(f"{categoria:<22} {cantidad:>4} movimientos")

Educacion                50 movimientos
Vivienda                 51 movimientos
Ingresos                 21 movimientos
Ahorro e inversion       47 movimientos
Ocio                     39 movimientos
Salud                    34 movimientos
Supermercado             39 movimientos
Transporte               38 movimientos
                         29 movimientos


_(tu respuesta acá)_

Sí aparece un categoría rara, una vacía al final con 29 movimientos que corresponde a las transacciones que no tenían categoría asignada en el archivo original.

<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 3.4 <span style='color:#BB3300; font-size:0.8em;'>** (10 pts)</span></h3>

Las categorías vacías se completan mirando la descripción del comercio. El diccionario de reglas ya
está armado, y `completar_categorias` también.

**Tienes que escribir `clasificar_por_palabra_clave(descripcion, reglas)` de forma distinta a la vista en clase en el Tema 3**:

- Recorre `reglas` y devuelve la **primera** categoría que tenga una palabra clave contenida en la
  descripción.
- Si ninguna coincide, devuelve la constante `SIN_CLASIFICAR`.
- Insensible a mayúsculas: `"Tienda Inglesa"` tiene que coincidir con `"tienda inglesa"`, esto se hace usando el método sobre strings .lower() que pasa todo el texto a minúscula, puedes probar en lugar de ello usar .upper() a ver que pasa.

Una vez que funcione, corré la celda que completa las categorías. **El objetivo es que quede cero
sin clasificar.**

Después contestá las dos preguntas:

1. ¿Cuántas categorías estaban vacías y cuántas quedaron sin clasificar?
2. El orden de las reglas importa, porque gana la primera que coincide. 
   Busca en este archivo un ejemplo concreto de una descripción que coincidiría con dos categorías distintas, o una palabra
   clave que es parte de otra palabra más larga y contesta qué pasaría si dieras vuelta el orden.

In [15]:
REGLAS_CATEGORIA = {
    "Supermercado": ["tienda inglesa", "devoto", "disco", "ta-ta", "macromercado"],
    "Vivienda": ["alquiler", "gastos comunes", "ute", "ose", "antel"],
    "Transporte": ["combustible", "ancap", "stm", "uber", "peaje", "service"],
    "Salud": ["farmacia", "mutualista", "odontologo", "optica"],
    "Educacion": ["universidad", "libros", "curso"],
    "Ingresos": ["salario", "aguinaldo", "freelance", "reintegro"],
    "Ahorro e inversion": ["caja de ahorro", "compra de ui", "fondo de inversion"],
    "Ocio": ["cine", "restaurante", "spotify", "netflix", "teatro"],
}

def completar_categorias(transacciones, reglas):
    """Rellena las categorías vacías y devuelve cuántas quedaron sin clasificar."""
    sin_clasificar = 0
    for transaccion in transacciones:
        if not transaccion["categoria"]:
            transaccion["categoria"] = clasificar_por_palabra_clave(
                transaccion["descripcion"], reglas
            )
            if transaccion["categoria"] == "Sin clasificar":
                sin_clasificar += 1
    return sin_clasificar

In [16]:
# Tu código acá: escribí clasificar_por_palabra_clave(descripcion, reglas)

SIN_CLASIFICAR = "Sin clasificar"


def clasificar_por_palabra_clave(descripcion, reglas):
    """Recorre las reglas y clasifica la descripción según la primera coincidencia de palabra clave."""
    descripcion_lower = descripcion.lower()

    for categoria, palabras_clave in reglas.items():
        for palabra in palabras_clave:
            if palabra.lower() in descripcion_lower:
                return categoria

    return SIN_CLASIFICAR

In [17]:
# Cuántas venían vacías, antes de tocar nada
vacias_antes = 0
for transaccion in transacciones:
    if not transaccion["categoria"]:
        vacias_antes += 1
print("Categorías vacías antes:", vacias_antes)

pendientes = completar_categorias(transacciones, REGLAS_CATEGORIA)
print("Quedaron sin clasificar:", pendientes)
print("Categorías finales:", sorted(contar_por_categoria(transacciones)))

Categorías vacías antes: 29
Quedaron sin clasificar: 0
Categorías finales: ['Ahorro e inversion', 'Educacion', 'Ingresos', 'Ocio', 'Salud', 'Supermercado', 'Transporte', 'Vivienda']


_(tus dos respuestas acá)_

1. Habían vacías 29 categorías, y terminaron quedando 0 sin clasificar.

2. En el archivo hay descripciones como "Transferencia caja de ahorro" (id: 10) que contienen la palabra clave "caja de ahorro" (categoría Ahorro e inversión); si diera vuelta el orden de las reglas y existiera otra categoría antes con la palabra clave "transferencia", la transacción se clasificaría incorrectamente en esa primera coincidencia.

<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 3.5 <span style='color:#BB3300; font-size:0.8em;'>** (8 pts)</span></h3>

Escribí `sumar_por_categoria(transacciones)`: devuelve un diccionario con el **monto total** de cada
categoría.

Es la misma estructura que `contar_por_categoria`, pero acumulando el monto
en lugar de contar.

Usala sobre los gastos para imprimir el gasto de cada categoría.

In [18]:
# Tu código acá
def sumar_por_categoria(transacciones):
    """Devuelve un diccionario con el monto total acumulado por cada categoría."""
    totales = {}

    for transaccion in transacciones:
        categoria = transaccion["categoria"]
        monto = transaccion["monto"]

        if categoria in totales:
            totales[categoria] += monto
        else:
            totales[categoria] = monto

    return totales


gastos_por_categoria = sumar_por_categoria(gastos)

print("Monto total gastado por categoría:")
for categoria, monto_total in gastos_por_categoria.items():
    print(f"{categoria:<22} ${abs(monto_total):,.0f}")

Monto total gastado por categoría:
Educacion              $75,321
Vivienda               $272,645
Ahorro e inversion     $151,590
Ocio                   $25,260
Salud                  $38,041
Supermercado           $97,994
Transporte             $34,798


<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 3.6 <span style='color:#BB3300; font-size:0.8em;'>** (8 pts)</span></h3>

Calcula la **tasa de ahorro** (sin utilizar una funcion) como la proporción de los ingresos que no se gastó (ingresos - gastos / ingresos) para todo el periodo y mostrala como porcentaje.

- Tene en cuenta que en este archivo los egresos vienen en **negativo**. 
- Si `total_ingresos` es cero, no se debe calcular la tasa, por lo que agrega un condicional con un mensaje de que en ese caso no hay calculo.

In [19]:
# Tu código acá
total_ingresos = 0.0
total_gastos = 0.0

for transaccion in transacciones:
    monto = transaccion["monto"]
    if monto > 0:
        total_ingresos += monto
    else:
        total_gastos += abs(monto)

if total_ingresos == 0:
    print("No hay calculo de la tasa de ahorro porque el total de ingresos es cero.")
else:
    tasa_ahorro = ((total_ingresos - total_gastos) / total_ingresos) * 100
    print(f"Total Ingresos: ${total_ingresos:,.2f}")
    print(f"Total Gastos: ${total_gastos:,.2f}")
    print(f"Tasa de ahorro: {tasa_ahorro:.2f}%")

Total Ingresos: $1,006,856.83
Total Gastos: $695,647.26
Tasa de ahorro: 30.91%


<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 3.7 <span style='color:#BB3300; font-size:0.8em;'>*** (12 pts)</span></h3>

Completa la funcion `resumir_mes(transacciones, mes)`. 

Devuelve un diccionario con tres claves: `ingresos`,
`gastos` y `balance` (ingresos restandole los gastos), calculadas **solo con los movimientos de ese mes**.

- Tene en cuenta que en este archivo los egresos vienen en **negativo**. 
- El mes llega como texto, `"2025-06"`. y se encuentra en el campo `mes`
- Agregar a la funcion que si el mes pedido no tiene ningún movimiento, imprimir mensaje "En el mes X no hay movimientos".

En la celda de abajo, mostrá el resumen de `"2025-06"` y el de `"2026-02"`, y probá la validación
pidiendo un mes que no existe.

In [20]:
# Completa el codigo

def resumir_por_mes(transacciones):
    """Devuelve un diccionario con los ingresos, gastos y balance agrupados por mes."""
    resumen = {}

    for transaccion in transacciones:
        mes = transaccion["fecha"][:7]
        monto = transaccion["monto"]

        if mes not in resumen:
            resumen[mes] = {"ingresos": 0.0, "gastos": 0.0}

        if monto > 0:
            resumen[mes]["ingresos"] += monto
        else:
            resumen[mes]["gastos"] += monto

    for mes in resumen:
        resumen[mes]["balance"] = resumen[mes]["ingresos"] + resumen[mes]["gastos"]

    return resumen


def resumir_mes(transacciones, mes_buscado):
    """Muestra el resumen de un mes específico o avisa si no hay registros."""
    resumen_general = resumir_por_mes(transacciones)

    if mes_buscado not in resumen_general:
        print(f"En el mes {mes_buscado} no hay movimientos.")
    else:
        datos = resumen_general[mes_buscado]
        print(f"Resumen del mes {mes_buscado}:")
        print(f"Ingresos: ${datos['ingresos']:,.2f}")
        print(f"Gastos:   ${abs(datos['gastos']):,.2f}")
        print(f"Balance:  ${datos['balance']:,.2f}\n")


resumir_mes(transacciones, "2025-06")
resumir_mes(transacciones, "2026-02")

# Prueba de validación con un mes qie no existe
resumir_mes(transacciones, "2024-01")

Resumen del mes 2025-06:
Ingresos: $79,335.25
Gastos:   $27,922.79
Balance:  $51,412.46

Resumen del mes 2026-02:
Ingresos: $50,332.48
Gastos:   $46,223.97
Balance:  $4,108.51

En el mes 2024-01 no hay movimientos.


<h2 style='font-weight:bold; color:#003366;'>Parte 4 : análisis <span style='color:#BB3300; font-size:0.8em;'>(16 puntos)</span></h2>

<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 4.1 <span style='color:#BB3300; font-size:0.8em;'>** (6 pts)</span></h3>

**¿En qué se gasta la plata?**

Con `filtrar_gastos` y `sumar_por_categoria`, armá una tabla del gasto total por categoría del
período completo, **ordenada de mayor a menor**, con el porcentaje que cada categoría representa del
gasto total.

Contestá:

1. Las tres categorías más grandes y qué porcentaje del gasto explican **juntas**.
2. Hay una categoría en esa lista que **no es consumo**: la plata no se fue de la cuenta, cambió de
   lugar. Decí cuál es, cuánto pesa, y recalculá el gasto total sin ella.

In [21]:
# Tu código acá
gastos = filtrar_gastos(transacciones)
gastos_por_cat = sumar_por_categoria(gastos)


gasto_total = abs(sum(gastos_por_cat.values()))


gastos_ordenados = sorted(
    gastos_por_cat.items(), key=lambda item: abs(item[1]), reverse=True
)


print(f"{'Categoría':<22} {'Monto ($)':>12} {'Porcentaje':>12}")
print("-" * 48)

for categoria, monto in gastos_ordenados:
    monto_pos = abs(monto)
    porcentaje = (monto_pos / gasto_total) * 100
    print(f"{categoria:<22} ${monto_pos:>11,.0f} {porcentaje:>11.2f}%")

print("-" * 48)
print(f"{'TOTAL':<22} ${gasto_total:>11,.0f} {'100.00%':>12}")

Categoría                 Monto ($)   Porcentaje
------------------------------------------------
Vivienda               $    272,645       39.19%
Ahorro e inversion     $    151,590       21.79%
Supermercado           $     97,994       14.09%
Educacion              $     75,321       10.83%
Salud                  $     38,041        5.47%
Transporte             $     34,798        5.00%
Ocio                   $     25,260        3.63%
------------------------------------------------
TOTAL                  $    695,647      100.00%


_(tus dos respuestas acá)_

1. Las tres categorías más grandes son Vivienda, Ahorro e inversión y Supermercado. Juntas representan el 75,07% del gastp del período, o sea tres cuartos del total es explicado solo por esas 3 categorías, siendo más de la mitad de ese 75,07% compuesto por el gasto en vivienda.

2. Categoría que no es consumo es Ahorro e inversión (que incluye movimientos como "compra de UI", "fondo de inversión" o "caja de ahorro"). No representa un gasto en consumo de la persona, sino una transferencia de activos dentro del propio patrimonio de esta.

In [22]:
monto_ahorro = abs(gastos_por_cat.get("Ahorro e inversion", 0.0))
gasto_real_consumo = gasto_total - monto_ahorro

porcentaje_ahorro = (monto_ahorro / gasto_total) * 100

print(f"\nMonto en Ahorro e inversión: ${monto_ahorro:,.0f} ({porcentaje_ahorro:.2f}% del total)")
print(f"Gasto total real en consumo:   ${gasto_real_consumo:,.0f}")


Monto en Ahorro e inversión: $151,590 (21.79% del total)
Gasto total real en consumo:   $544,058


<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 4.2 <span style='color:#BB3300; font-size:0.8em;'>*** (10 pts)</span></h3>

**¿Cómo evolucionó la tasa de ahorro mes a mes?**

Con `resumir_mes` y `calcular_tasa_ahorro` (la funcion que usamos en la clase que esta abajo), armá una tabla con una fila por mes: ingresos, gastos,
balance y tasa de ahorro. Son 18 meses. La lista de meses la sacás de los datos, no escrita a mano.

Contestá:

1. El mes con la **mejor** tasa de ahorro y el mes con la **peor**, con sus cuatro números.
2. Hay **tres meses** cuyo ingreso no es comparable con el resto. Decí cuáles son, por qué, y qué
   pasa con la tasa de ahorro de esos meses.

In [23]:
def calcular_tasa_ahorro(total_ingresos, total_gastos):
    if total_ingresos == 0:
        print("No se puede calcular la tasa de ahorro sin ingresos")
        return None
    return (total_ingresos - total_gastos) / total_ingresos

# 1. Obtener el resumen mensual de las transacciones
resumen_mensual = resumir_por_mes(transacciones)

# 2. Obtener la lista de meses ordenada cronológicamente a partir de los datos
meses_ordenados = sorted(resumen_mensual.keys())

# 3. Imprimir la tabla mes a mes
print(
    f"{'Mes':<10} {'Ingresos ($)':>14} {'Gastos ($)':>14} {'Balance ($)':>14} {'Tasa Ahorro':>14}"
)
print("-" * 70)

mejores_peores_meses = []

for mes in meses_ordenados:
    datos = resumen_mensual[mes]
    ingresos = datos["ingresos"]
    # Convertimos los gastos a positivo para calcular e imprimir de forma clara
    gastos = abs(datos["gastos"])
    balance = datos["balance"]

    tasa = calcular_tasa_ahorro(ingresos, gastos)

    str_tasa = f"{tasa * 100:>13.2f}%" if tasa is not None else "           N/A"
    print(
        f"{mes:<10} ${ingresos:>13,.2f} ${gastos:>13,.2f} ${balance:>13,.2f} {str_tasa}"
    )

    if tasa is not None:
        mejores_peores_meses.append((tasa, mes, ingresos, gastos, balance))

# 4. Encontrar el mes con mejor y peor tasa
mejores_peores_meses.sort(key=lambda x: x[0], reverse=True)

mejor = mejores_peores_meses[0]
peor = mejores_peores_meses[-1]

print("\n--- RESUMEN TASA DE AHORRO ---")
print(
    f"Mejor mes: {mejor[1]} -> Ingresos: ${mejor[2]:,.2f} | Gastos: ${mejor[3]:,.2f} | Balance: ${mejor[4]:,.2f} | Tasa: {mejor[0]*100:.2f}%"
)
print(
    f"Peor mes:  {peor[1]} -> Ingresos: ${peor[2]:,.2f} | Gastos: ${peor[3]:,.2f} | Balance: ${peor[4]:,.2f} | Tasa: {peor[0]*100:.2f}%"
)

Mes          Ingresos ($)     Gastos ($)    Balance ($)    Tasa Ahorro
----------------------------------------------------------------------
2025-01    $    50,572.38 $    37,423.64 $    13,148.74         26.00%
2025-02    $    50,356.72 $    32,718.59 $    17,638.13         35.03%
2025-03    $    50,294.71 $    33,083.28 $    17,211.43         34.22%
2025-04    $    53,003.14 $    43,281.03 $     9,722.11         18.34%
2025-05    $    52,389.11 $    49,594.89 $     2,794.22          5.33%
2025-06    $    79,335.25 $    27,922.79 $    51,412.46         64.80%
2025-07    $    50,000.97 $    43,790.79 $     6,210.18         12.42%
2025-08    $    53,921.70 $    34,306.56 $    19,615.14         36.38%
2025-09    $    52,384.57 $    42,645.78 $     9,738.79         18.59%
2025-10    $    53,298.85 $    45,056.97 $     8,241.88         15.46%
2025-11    $    52,692.05 $    26,030.71 $    26,661.34         50.60%
2025-12    $    78,085.12 $    51,212.96 $    26,872.16         34.41%
2026-0

_(tus dos respuestas acá)_

1. Mejor y peor mes con sus cuatro números:
- Mejor mes (2025-06): Ingresos de $79,335.25, Gastos de $27,922.79, Balance de $51,412.46 y Tasa de ahorro del 64.80%.
- Peor mes (2025-05): Ingresos de $52,389.11, Gastos de $49,594.89, Balance de $2,794.22 y Tasa de ahorro del 5.33%.


2. Los tres meses con ingresos no comparables son Junio 2025 (2025-06), Diciembre 2025 (2025-12) y Junio 2026 (2026-06).
Esto es porque son los meses en los que se gana el aguinaldo, haciendo que los ingresos sean más altos que lo normal (subiendo de los alrededor de 50mil-53mil habituales a más de 73mil–79mil).
- Al dispararse el nivel de ingresos mientras la estructura de gastos se mantiene más o menos normal, la tasa de ahorro sube de forma atípica (alcanzando su máximo de 64.80% en 2025-06 y 47.24% en 2026-06).

# Nicolás Gómez